# Diffusion foundations: learning to remove noise

Diffusion models turn generation into a sequence of denoising problems. A clean sample is gradually corrupted by known noise; a neural network learns to predict the noise at a randomly chosen time step. Sampling then runs that process in reverse.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn

sns.set_theme(style='whitegrid', context='notebook')
torch.manual_seed(5); np.random.seed(5)

## The forward process

For a simple one-dimensional illustration, the noisy sample at time `t` is

`x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) epsilon`,

where `epsilon` is standard Gaussian noise. The schedule controls how quickly the signal is forgotten.

In [ ]:
T = 50
beta = torch.linspace(1e-4, 0.02, T)
alpha = 1 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
x0 = torch.linspace(-1, 1, 400)
epsilon = torch.randn_like(x0)
steps = [0, 4, 12, 24, 49]
fig, axes = plt.subplots(1, len(steps), figsize=(14, 3), sharey=True)
for ax, t in zip(axes, steps):
    xt = alpha_bar[t].sqrt() * x0 + (1 - alpha_bar[t]).sqrt() * epsilon
    ax.scatter(x0.numpy(), xt.numpy(), s=4, alpha=.35)
    ax.set_title(f't={t+1}')
    ax.set_xlabel('clean x₀')
axes[0].set_ylabel('noisy xₜ'); plt.tight_layout()

## A tiny noise predictor

The network receives the noisy value and a representation of the time step. It predicts the noise that was added. The training target is available because the forward corruption process is known; no human label is required.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, width):
        super().__init__(); self.projection = nn.Linear(1, width)
    def forward(self, t):
        return torch.sin(self.projection(t.float().unsqueeze(1)))

class NoisePredictor(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.time = TimeEmbedding(width)
        self.network = nn.Sequential(nn.Linear(1 + width, width), nn.SiLU(), nn.Linear(width, 1))
    def forward(self, x, t):
        return self.network(torch.cat([x.unsqueeze(1), self.time(t / T)], dim=1)).squeeze(1)

model = NoisePredictor()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(1200):
    t = torch.randint(0, T, (128,))
    clean = torch.randn(128) * 0.7 + torch.where(torch.rand(128) > .5, 1.0, -1.0)
    noise = torch.randn_like(clean)
    noisy = alpha_bar[t].sqrt() * clean + (1 - alpha_bar[t]).sqrt() * noise
    loss = nn.functional.mse_loss(model(noisy, t), noise)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
print(f'final noise-prediction loss: {loss.item():.4f}')

The real image version replaces the scalar network with a U-Net. Its encoder collects context at lower resolutions, its decoder restores spatial detail, and skip connections preserve local information. The objective remains the same: predict noise (or an equivalent parameterization) conditioned on the time step.